<a href="https://colab.research.google.com/github/antoniomt846-max/Clase-intro/blob/main/Taller_Graficos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller · De los datos a un dashboard publicado

**Programación — ADMI 1301**

Trabajarás con las **2.000 ventas de una cadena de papelerías en 2025**. En este taller vas a:

1. **Graficar** (6 ejercicios con `groupby`, `crosstab` y `pivot_table`).
2. **Organizar** tus gráficos en una sola figura con `subplots`: tu *dashboard*.
3. **Embellecerlo con prompts** para Gemini, en el mismo Colab.
4. **Elegir tu versión favorita**, escribir tus conclusiones y **publicar tu notebook en GitHub**.

Las instrucciones completas están en la guía en Word. Aquí encuentras lo esencial de cada ejercicio y una **verificación** (`assert`) para saber si vas bien.

## 0. Preparación
Los datos ya vienen **dentro del notebook**: no tienes que subir ningún archivo.

1. La primera celda escribe `ventas_papelerias_2025.csv` (2.000 ventas, 13 columnas) en Colab.
2. La segunda celda lo carga en el DataFrame `df` y crea la columna `mes`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

df = pd.read_csv("ventas_papelerias_2025.csv")
df["fecha"] = pd.to_datetime(df["fecha"])
df["mes"] = df["fecha"].dt.month        # 1 = enero ... 12 = diciembre

assert df.shape == (2000, 14), "Revisa que la celda de los datos se haya ejecutado completa"
df.head()

**Conoce tus datos** antes de graficar: cuántas filas hay, qué columnas, qué valores tiene cada categoría.

In [ ]:
# Si se reinició el entorno y df no existe, se recargan los datos (escritos en la Sección 0)
if "df" not in globals():
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    df = pd.read_csv("ventas_papelerias_2025.csv")
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["mes"] = df["fecha"].dt.month

print(df.shape)
df.info()
for col in ["ciudad", "canal", "segmento", "categoria", "metodo_pago"]:
    print(col, "->", sorted(df[col].unique()))

---
# Parte A · Seis gráficos
En cada ejercicio: **(1)** prepara una tabla con `groupby`, `crosstab` o `pivot_table`, **(2)** guárdala en la variable indicada, **(3)** grafícala. Ponle **título** a cada gráfico.

### Ejercicio 1 · Top 5 de productos *(barras horizontales)*
**Contexto.** La gerente de compras debe decidir en qué productos concentrar el inventario del próximo año.

**Tarea.** Calcula los ingresos (`total`) por `producto`, quédate con los **5 mayores** y grafícalos en barras horizontales. Guarda la tabla en **`top_productos`**.

*Herramientas:* `groupby`, `sum`, `sort_values`, `head`, `.plot.barh()`

In [ ]:
# Si se reinició el entorno y df no existe, se recargan los datos (escritos en la Sección 0)
if "df" not in globals():
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    df = pd.read_csv("ventas_papelerias_2025.csv")
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["mes"] = df["fecha"].dt.month

# Ejercicio 1
# (1) Ingresos por producto, ordenados de mayor a menor, y nos quedamos con los 5 primeros
top_productos = (df.groupby("producto")["total"]
                   .sum()
                   .sort_values(ascending=False)
                   .head(5))
display(top_productos)

# (2) Gráfico: sort_values() de menor a mayor para que el mayor quede arriba
ax = top_productos.sort_values().plot.barh(figsize=(8, 4), color="steelblue")
ax.set_title("Top 5 productos por ingresos · 2025")
ax.set_xlabel("Ingresos (pesos)")
ax.set_ylabel("Producto")
plt.show()

# Verificación
assert top_productos.index[0] == "Cuaderno" and len(top_productos) == 5
print("Ejercicio 1 correcto ✔")

### Ejercicio 2 · Ingresos mes a mes: Bogotá vs Medellín *(líneas)*
**Contexto.** La dirección regional quiere comparar la evolución de sus dos ciudades principales durante el año.

**Tarea.** **Selecciona** solo las ventas de Bogotá y Medellín (`isin`). Con una **tabla dinámica** pon los meses en las filas, las ciudades en las columnas y la suma de `total` como valores. Guárdala en **`ing_mes_ciudad`** y grafícala como líneas con marcadores.

*Herramientas:* filtrado con `isin`, `pivot_table`, `.plot.line(marker="o")`

In [ ]:
# Si se reinició el entorno y df no existe, se recargan los datos (escritos en la Sección 0)
if "df" not in globals():
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    df = pd.read_csv("ventas_papelerias_2025.csv")
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["mes"] = df["fecha"].dt.month

# Ejercicio 2
# (1) Filtro: solo Bogotá y Medellín
bog_med = df[df["ciudad"].isin(["Bogotá", "Medellín"])]

# (2) Tabla dinámica: meses en filas, ciudades en columnas, suma de total
ing_mes_ciudad = bog_med.pivot_table(index="mes", columns="ciudad",
                                     values="total", aggfunc="sum")
display(ing_mes_ciudad)

# (3) Gráfico de líneas con marcadores
ax = ing_mes_ciudad.plot.line(marker="o", figsize=(9, 4))
ax.set_title("Ingresos mensuales: Bogotá vs Medellín · 2025")
ax.set_xlabel("Mes")
ax.set_ylabel("Ingresos (pesos)")
ax.set_xticks(range(1, 13))
plt.show()

# Verificación
assert ing_mes_ciudad.shape == (12, 2)
print("Ejercicio 2 correcto ✔")

### Ejercicio 3 · ¿Por qué canal se vende más? *(gráfico de pie)*
**Contexto.** La empresa evalúa si invierte más en su tienda online o en la atención telefónica.

**Tarea.** Calcula los ingresos totales por `canal`, guárdalos en **`ing_canal`** y grafica la participación de cada canal con un gráfico de pie que muestre los **porcentajes con un decimal**.

*Herramientas:* `groupby`, `.plot.pie(autopct=...)`

In [ ]:
# Si se reinició el entorno y df no existe, se recargan los datos (escritos en la Sección 0)
if "df" not in globals():
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    df = pd.read_csv("ventas_papelerias_2025.csv")
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["mes"] = df["fecha"].dt.month

# Ejercicio 3
# (1) Ingresos por canal
ing_canal = df.groupby("canal")["total"].sum()
display(ing_canal)

# (2) Pie con porcentajes de un decimal
ax = ing_canal.plot.pie(autopct="%1.1f%%", figsize=(5, 5), startangle=90)
ax.set_title("Participación de los ingresos por canal · 2025")
ax.set_ylabel("")   # quitamos la etiqueta "total" que pone pandas
plt.show()

# Verificación
assert ing_canal["Telefónico"] / ing_canal.sum() > 0.5
print("Ejercicio 3 correcto ✔")

### Ejercicio 4 · ¿Cada segmento compra lo mismo? *(barras apiladas)*
**Contexto.** El área comercial sospecha que los clientes corporativos compran productos distintos a los minoristas.

**Tarea.** Con `pd.crosstab` cuenta las ventas por `segmento` (filas) y `categoria` (columnas), **normalizadas por fila** y expresadas en **porcentaje** (de 0 a 100). Guarda la tabla en **`tabla_seg_cat`** y grafícala en barras **apiladas**.

*Herramientas:* `pd.crosstab(..., normalize="index")`, `.plot.bar(stacked=True)`

In [ ]:
# Si se reinició el entorno y df no existe, se recargan los datos (escritos en la Sección 0)
if "df" not in globals():
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    df = pd.read_csv("ventas_papelerias_2025.csv")
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["mes"] = df["fecha"].dt.month

# Ejercicio 4
# (1) Crosstab: segmento en filas, categoría en columnas, normalizado por fila y en %
tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100
display(tabla_seg_cat.round(1))

# (2) Barras apiladas
ax = tabla_seg_cat.plot.bar(stacked=True, figsize=(8, 5))
ax.set_title("Mezcla de categorías por segmento (% de ventas)")
ax.set_xlabel("Segmento")
ax.set_ylabel("% de las ventas del segmento")
ax.legend(title="Categoría", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.show()

# Verificación
assert tabla_seg_cat.shape == (3, 5)
assert (tabla_seg_cat.sum(axis=1).round(6) == 100).all()
print("Ejercicio 4 correcto ✔")

### Ejercicio 5 · Productividad de los vendedores *(dispersión)*
**Contexto.** Recursos humanos revisa qué vendedores concretan más ventas y cuáles venden tickets más altos.

**Tarea.** Con `groupby` por `vendedor` y `ciudad`, calcula para cada vendedor el **número de ventas** (`n_ventas`) y el **valor promedio de sus ventas** (`ticket_prom`). Guarda la tabla en **`vend`** (con índice normal: usa `reset_index()`). Haz una dispersión (`n_ventas` en X, `ticket_prom` en Y) con un **color por ciudad** y leyenda.

*Herramientas:* `groupby().agg(nombre=("columna", "función"))`, `reset_index`, `plt.scatter` dentro de un `for`

In [ ]:
# Si se reinició el entorno y df no existe, se recargan los datos (escritos en la Sección 0)
if "df" not in globals():
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    df = pd.read_csv("ventas_papelerias_2025.csv")
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["mes"] = df["fecha"].dt.month

# Ejercicio 5
# (1) Número de ventas y ticket promedio por vendedor (y su ciudad)
vend = (df.groupby(["vendedor", "ciudad"])
          .agg(n_ventas=("id_venta", "count"),
               ticket_prom=("total", "mean"))
          .reset_index())
display(vend.sort_values("n_ventas", ascending=False))

# (2) Dispersión con un color por ciudad
plt.figure(figsize=(8, 5))
for ciudad, grupo in vend.groupby("ciudad"):
    plt.scatter(grupo["n_ventas"], grupo["ticket_prom"], label=ciudad, s=80)
plt.title("Productividad de los vendedores · 2025")
plt.xlabel("Número de ventas")
plt.ylabel("Ticket promedio (pesos)")
plt.legend(title="Ciudad")
plt.show()

# Verificación
assert len(vend) == 12 and vend["n_ventas"].sum() == 2000
print("Ejercicio 5 correcto ✔")

### Ejercicio 6 · ¿Qué tan grandes son las ventas de cada segmento? *(diagrama de cajas)*
**Contexto.** Para planear el segundo semestre, finanzas quiere ver cómo se distribuye el valor de las ventas por tipo de cliente.

**Tarea.** **Selecciona** solo las ventas del **segundo semestre** (`mes` de 7 a 12) y guárdalas en **`sem2`**. Haz un diagrama de cajas del `total` por `segmento`, **sin mostrar los valores atípicos** (`showfliers=False`).

*Herramientas:* filtro con `>=`, `.boxplot(column=..., by=...)`

In [ ]:
# Si se reinició el entorno y df no existe, se recargan los datos (escritos en la Sección 0)
if "df" not in globals():
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    df = pd.read_csv("ventas_papelerias_2025.csv")
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["mes"] = df["fecha"].dt.month

# Ejercicio 6
# (1) Filtro: segundo semestre (julio a diciembre)
sem2 = df[df["mes"] >= 7]
print("Ventas del segundo semestre:", len(sem2))

# (2) Diagrama de cajas del total por segmento, sin atípicos
ax = sem2.boxplot(column="total", by="segmento", showfliers=False, figsize=(8, 5))
plt.suptitle("")   # quitamos el título automático de pandas
ax.set_title("Valor de las ventas por segmento · 2.º semestre 2025")
ax.set_xlabel("Segmento")
ax.set_ylabel("Total de la venta (pesos)")
plt.show()

# Verificación
assert len(sem2) == 1050
print("Ejercicio 6 correcto ✔")

---
# Parte B · Todo en una sola figura (`subplots`)
### Ejercicio 7 · Organiza tus 6 gráficos en una cuadrícula de 2 × 3
**Contexto.** La gerente quiere ver todo en **una sola imagen** para presentarla en la junta.

**Tarea.** Crea una figura con `fig, axs = plt.subplots(2, 3, figsize=(18, 10))` y dibuja cada gráfico en su posición pasando **`ax=axs[fila, columna]`**. Agrega un título general con `fig.suptitle(...)` y termina con `plt.tight_layout()`.

**Importante:** escribe el código en la celda que empieza con `%%writefile figura_base.py` (así el programa guarda tu código en un archivo, que usarás en la Parte C). Luego ejecuta la celda siguiente (`%run -i figura_base.py`) para ver la figura.

*Pistas:* `plot.barh(ax=axs[0, 0])` · `plot.pie(...)` también acepta `ax` · para el boxplot: `boxplot(column=..., by=..., ax=...)` · para la dispersión: `axs[1, 1].scatter(...)`

In [ ]:
%%writefile figura_base.py
# Ejercicio 7: código de la figura 2x3
# Las tablas se recalculan aquí para que el archivo funcione solo (Gemini lo ejecuta aparte, solo con df)

top_productos = df.groupby("producto")["total"].sum().sort_values(ascending=False).head(5)
ing_mes_ciudad = (df[df["ciudad"].isin(["Bogotá", "Medellín"])]
                  .pivot_table(index="mes", columns="ciudad", values="total", aggfunc="sum"))
ing_canal = df.groupby("canal")["total"].sum()
tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100
vend = (df.groupby(["vendedor", "ciudad"])
          .agg(n_ventas=("id_venta", "count"), ticket_prom=("total", "mean"))
          .reset_index())
sem2 = df[df["mes"] >= 7]

fig, axs = plt.subplots(2, 3, figsize=(18, 10))

# [0, 0] Top 5 productos
top_productos.sort_values().plot.barh(ax=axs[0, 0], title="Top 5 productos por ingresos")
axs[0, 0].set_xlabel("Ingresos (pesos)")
axs[0, 0].set_ylabel("")

# [0, 1] Bogotá vs Medellín mes a mes
ing_mes_ciudad.plot.line(ax=axs[0, 1], marker="o", title="Ingresos mensuales: Bogotá vs Medellín")
axs[0, 1].set_xlabel("Mes")
axs[0, 1].set_ylabel("Ingresos (pesos)")
axs[0, 1].set_xticks(range(1, 13))

# [0, 2] Participación por canal
ing_canal.plot.pie(ax=axs[0, 2], autopct="%1.1f%%", startangle=90, title="Ingresos por canal")
axs[0, 2].set_ylabel("")

# [1, 0] Mezcla de categorías por segmento
tabla_seg_cat.plot.bar(ax=axs[1, 0], stacked=True, rot=0, title="Categorías por segmento (%)")
axs[1, 0].set_xlabel("Segmento")
axs[1, 0].set_ylabel("% de las ventas")
axs[1, 0].set_ylim(0, 125)   # espacio arriba para la leyenda
axs[1, 0].legend(title="Categoría", fontsize=8, ncol=3, loc="upper center")

# [1, 1] Dispersión de vendedores
for ciudad, grupo in vend.groupby("ciudad"):
    axs[1, 1].scatter(grupo["n_ventas"], grupo["ticket_prom"], label=ciudad, s=70)
axs[1, 1].set_title("Productividad de los vendedores")
axs[1, 1].set_xlabel("Número de ventas")
axs[1, 1].set_ylabel("Ticket promedio (pesos)")
axs[1, 1].legend(title="Ciudad", fontsize=8)

# [1, 2] Boxplot del 2.º semestre
sem2.boxplot(column="total", by="segmento", showfliers=False, ax=axs[1, 2])
axs[1, 2].set_title("Valor de las ventas por segmento (2.º sem.)")
axs[1, 2].set_xlabel("Segmento")
axs[1, 2].set_ylabel("Total de la venta (pesos)")

fig.suptitle("Ventas de la cadena de papelerías · 2025", fontsize=18)   # reemplaza el título automático del boxplot
plt.tight_layout()

In [ ]:
# Si se reinició el entorno y df no existe, se recargan los datos (escritos en la Sección 0)
if "df" not in globals():
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    df = pd.read_csv("ventas_papelerias_2025.csv")
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["mes"] = df["fecha"].dt.month

%run -i figura_base.py

In [ ]:
# ✅ Verificación
assert len(fig.axes) >= 6, f"La figura debe tener 6 gráficos, tiene {len(fig.axes)}"
print("Ejercicio 7 correcto ✔")

---
# Parte C · Hazla ver bonita con prompts
Ahora **tú escribes el prompt y Gemini reescribe el código** de tu figura para que se vea profesional, casi como un dashboard.


### Herramientas *(ya están hechas)*
Ejecuta esta celda. Define dos funciones:

- **`embellecer(prompt, nombre)`**: envía tu prompt **junto con tu código de `figura_base.py`**, ejecuta el código nuevo que devuelve Gemini y muestra la figura. Cada versión queda guardada en `VERSIONES[nombre]`.
- **`mostrar(nombre)`**: vuelve a dibujar una versión guardada.

In [ ]:
# Conexión con Gemini: la API key se lee desde los Secrets de Colab (ícono de llave, nombre GEMINI_API_KEY).
# Nunca la escribas en una celda: el notebook terminará en un repositorio público.
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
print("Cliente de Gemini listo ✔")

In [ ]:
import re

VERSIONES = {}      # aquí se guarda el código de cada versión ("v1", "v2", ...)

REGLAS_FIGURA = """

Reglas técnicas obligatorias:
- Usa solo pandas y matplotlib (y numpy si lo necesitas). No instales otras librerías.
- Los datos ya están cargados en el DataFrame `df` (columnas: id_venta, fecha, ciudad, canal, segmento, vendedor,
  producto, categoria, metodo_pago, cantidad, precio_unitario, descuento, total, mes). No cargues archivos.
- Conserva los MISMOS 6 gráficos y los mismos datos del código actual (puedes recalcular las tablas con el mismo código);
  cambia solo la presentación.
- La figura debe quedar en una variable llamada `fig` (2 filas x 3 columnas).
- NO uses plt.show() ni guardes archivos.
- Entrega solo UN bloque de código Python completo.
"""


def extraer_codigo(texto):
    """Devuelve el contenido del primer bloque de código ``` de la respuesta."""
    bloque = re.search(r"```(?:python)?\s*\n(.*?)```", texto, re.S)
    return bloque.group(1) if bloque else texto


def dibujar(codigo):
    """Ejecuta el código de una figura y la muestra."""
    import numpy as np
    plt.close("all")
    espacio = {"df": df, "pd": pd, "plt": plt, "np": np}
    exec(codigo, espacio)
    figura = espacio["fig"]
    display(figura)
    plt.close(figura)


def embellecer(prompt, nombre, modelo="gemini-flash-latest"):
    """Gemini reescribe tu figura según tu prompt; se ejecuta, se muestra y se guarda en VERSIONES."""
    codigo_actual = open("figura_base.py", encoding="utf-8").read()
    contenido = f"{prompt}\n\nEste es mi código actual:\n```python\n{codigo_actual}\n```{REGLAS_FIGURA}"
    respuesta = client.models.generate_content(model=modelo, contents=contenido)
    codigo = extraer_codigo(respuesta.text)
    try:
        dibujar(codigo)
    except Exception as e:
        print("❌ El código de Gemini falló:", type(e).__name__, e)
        print("Copia ese error en tu siguiente prompt y pídele que lo corrija.")
        return
    VERSIONES[nombre] = codigo
    print(f"✔ Versión '{nombre}' guardada")


def mostrar(nombre):
    """Vuelve a dibujar una versión guardada."""
    dibujar(VERSIONES[nombre])

### Ejercicio 8 · Tu primer prompt de estilo
Completa las secciones de `prompt_v1`. Sé **específico**: "usa una paleta de azules y naranja" funciona mejor que "hazlo bonito".

- **PÚBLICO Y OBJETIVO:** quién verá la figura y para qué.
- **PALETA Y TIPOGRAFÍA:** colores (por ciudad, por producto…), fondo, tamaños de letra.
- **TÍTULOS:** que cada gráfico tenga un título que diga la **conclusión** (por ejemplo, "Bogotá lidera las ventas"), no solo el nombre.
- **EJES Y NÚMEROS:** formatos con separador de miles o "$ 12 M", etiquetas de ejes claras.
- **LIMPIEZA VISUAL:** quitar bordes y marcos innecesarios, cuadrícula suave, valores sobre las barras.

In [ ]:
# Si se reinició el entorno y df no existe, se recargan los datos (escritos en la Sección 0)
if "df" not in globals():
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    df = pd.read_csv("ventas_papelerias_2025.csv")
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["mes"] = df["fecha"].dt.month

prompt_v1 = """
PÚBLICO Y OBJETIVO: La figura la verá la gerente general y la junta directiva de una cadena de papelerías,
proyectada en una pantalla durante la junta. Debe servir para entender en menos de un minuto qué productos,
canales, segmentos y vendedores explican los ingresos de 2025. Tiene que parecer un dashboard profesional.

PALETA Y TIPOGRAFÍA: Fondo de la figura gris muy claro (#F4F5F7) y paneles blancos. Un color fijo por ciudad
en todos los gráficos: Bogotá azul oscuro (#1F3A93), Medellín naranja (#F39C12), Cali verde (#27AE60),
Barranquilla amarillo (#F1C40F), Bucaramanga violeta (#8E44AD). Para gráficos sin ciudad usa el azul oscuro
como color principal y grises para lo secundario; en el pie usa azul oscuro para Telefónico y dos tonos de gris
azulado para Online y Tienda. Para las 5 categorías usa 5 tonos de una misma paleta de azules y verdes.
Fuente sans-serif (DejaVu Sans). Títulos de cada panel en negrita, tamaño 13, alineados a la izquierda.
Título general tamaño 20 en negrita y un subtítulo gris tamaño 12 que diga "2.000 ventas · enero a diciembre de 2025".

TÍTULOS: Cada panel debe tener un título que diga la conclusión, no solo el nombre:
- Top 5: "El Cuaderno lidera los ingresos"
- Líneas: "Bogotá vende más que Medellín casi todo el año"
- Pie: "El canal Telefónico trae más de la mitad de los ingresos"
- Barras apiladas: "Los corporativos compran más Papel y Organización"
- Dispersión: "Laura Gómez es la que más ventas concreta"
- Cajas: "Mayoristas y corporativos hacen las ventas más grandes"

EJES Y NÚMEROS: Muestra los ingresos en millones con el formato "$ 20,7 M" (usa FuncFormatter).
En el boxplot y en el ticket promedio usa miles: "$ 60 mil". En el eje X del gráfico de líneas usa los nombres
cortos de los meses (Ene, Feb, ..., Dic). Etiquetas de ejes claras y en español; en las barras apiladas el eje Y
va de 0 a 100 con el símbolo %.

LIMPIEZA VISUAL: Quita los bordes superior y derecho de todos los paneles. Cuadrícula suave (gris claro,
línea punteada) solo en el eje de los valores. Escribe el valor encima o al final de cada barra del Top 5.
Leyendas sin marco; en la dispersión la leyenda fuera del área de los puntos. Quita el título automático que
pone pandas en el boxplot ("Boxplot grouped by..."). Sin superposiciones: usa tight_layout o constrained_layout.
"""

embellecer(prompt_v1, "v1")

### Ejercicio 9 · Itera: mejora tu prompt
Mira la figura que obtuviste: ¿qué no te gusta? Agrega una sección **AJUSTES** con **al menos dos cambios concretos** (por ejemplo: otra paleta, letras más grandes, anotar el valor máximo del gráfico de líneas, cambiar el pie por una dona…). Es la **iteración de prompts** que vimos en clase.

In [ ]:
# Si se reinició el entorno y df no existe, se recargan los datos (escritos en la Sección 0)
if "df" not in globals():
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    df = pd.read_csv("ventas_papelerias_2025.csv")
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["mes"] = df["fecha"].dt.month

# Antes de ejecutar: mira la v1 y ajusta estos cambios a lo que tú observaste
prompt_v2 = prompt_v1 + """

AJUSTES:
1. Cambia el gráfico de pie por una dona (wedgeprops con width=0.4) y escribe en el centro
   "$ 118,7 M" (el total de ingresos del año) en negrita.
2. En el gráfico de líneas anota con una flecha el mes de mayor ingreso de Bogotá, con su valor en "$ X,X M".
3. Aumenta el tamaño de las etiquetas de los ejes a 11 y el de los números de los ejes a 10, para que se lean
   bien al proyectar.
4. En la dispersión escribe el nombre de cada vendedor junto a su punto en letra pequeña (tamaño 8).
5. En las barras apiladas escribe el porcentaje dentro de cada segmento de barra solo si es mayor a 8 %,
   en blanco, sin decimales.
Conserva exactamente los mismos 6 gráficos y las mismas cifras de la versión anterior.
"""

embellecer(prompt_v2, "v2")

### Ejercicio 10 *(opcional)* · Una tercera versión
Si quieres seguir mejorando, crea `prompt_v3` (parte de `prompt_v2`) y ejecútalo con el nombre `"v3"`. Tendrás tres versiones para comparar.

In [ ]:
# Si se reinició el entorno y df no existe, se recargan los datos (escritos en la Sección 0)
if "df" not in globals():
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    df = pd.read_csv("ventas_papelerias_2025.csv")
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["mes"] = df["fecha"].dt.month

prompt_v3 = prompt_v2 + """

AJUSTES FINALES:
1. Usa un solo color de énfasis por panel: resalta en azul oscuro solo el elemento líder (el Cuaderno,
   el canal Telefónico, Laura Gómez) y deja el resto en gris claro, para que la conclusión salte a la vista.
2. Agrega una franja superior con tres indicadores grandes antes de los gráficos:
   "Ingresos 2025: $ 118,7 M", "Ventas: 2.000" y "Ticket promedio: $ 59 mil" (puedes usar fig.text,
   sin cambiar la cuadrícula de 2 x 3).
3. Si alguna leyenda tapa los datos, muévela debajo del gráfico en una sola fila.
"""
embellecer(prompt_v3, "v3")

In [ ]:
# ✅ Verificación
assert len(VERSIONES) >= 2, "Debes tener al menos dos versiones de tu figura (v1 y v2)"
print("Versiones guardadas:", list(VERSIONES), "· Parte C correcta ✔")

---
# Parte D · Elige tu favorita y publícala
### Ejercicio 11 · Tu dashboard final y tus conclusiones
Mira tus versiones y **elige la que más te guste**. Luego escribe **3 hallazgos**: frases cortas con **cifras reales** sacadas de tus tablas. Se mostrarán debajo de tu dashboard.

*Ejemplo de hallazgo:* "Bogotá genera el 32 % de los ingresos de la cadena."

In [ ]:
figura_elegida = "v2"      # cambia por "v1", "v2" o "v3" según la que más te guste

hallazgos = [
    "El canal Telefónico genera el 51,8 % de los ingresos ($ 61,5 M de $ 118,7 M); Online apenas el 19,3 %.",
    "El Cuaderno es el producto líder con $ 20,7 M, casi $ 6,4 M más que el Block de notas ($ 14,3 M).",
    "Los clientes Corporativos compran distinto: el 38,2 % de sus ventas es Papel, frente al 15,2 % de los Minoristas.",
]

In [ ]:
# ✅ Verificación
assert figura_elegida in VERSIONES, f"No existe la versión {figura_elegida}: revisa figura_elegida"
assert len(hallazgos) == 3 and all(len(h) > 25 and "ESCRIBE AQUÍ" not in h for h in hallazgos), "Escribe tus 3 hallazgos con cifras"
print("Ejercicio 11 correcto ✔")

### Tu dashboard final
Ejecuta la celda: muestra la versión que elegiste y tus tres hallazgos. **Esta es la salida que se verá en GitHub.**

In [ ]:
# Si se reinició el entorno y df no existe, se recargan los datos (escritos en la Sección 0)
if "df" not in globals():
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    df = pd.read_csv("ventas_papelerias_2025.csv")
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["mes"] = df["fecha"].dt.month

mostrar(figura_elegida)

print("Hallazgos:")
for h in hallazgos:
    print("•", h)

### Ejercicio 12 · Publica tu notebook en GitHub
GitHub muestra los notebooks **con sus figuras**, así que tu notebook publicado es tu dashboard. Solo necesitas una cuenta gratuita en [github.com](https://github.com).

**Antes de publicar:**
- Ejecuta todo el notebook de arriba abajo (*Entorno de ejecución → Ejecutar todo*) para que las figuras queden guardadas en las salidas.
- **Nunca escribas tu API key dentro de una celda.** Los *Secrets* de Colab no se guardan en el notebook y por eso son seguros; el repositorio será público.

**Pasos:**
1. **Crea el repositorio:** botón **New** → nombre `mi-dashboard-ventas` → **Public** → *Create repository*.
2. **Guarda el notebook en GitHub.** Opción A: en Colab, *Archivo → Guardar una copia en GitHub*, autoriza, elige tu repositorio y acepta. Opción B: *Archivo → Descargar → .ipynb* y, en el repositorio, **Add file → Upload files** y arrastra el archivo.
3. **Abre el notebook en el repositorio** (haz clic en el archivo `.ipynb`): GitHub lo dibuja con tus figuras.
4. **Copia el enlace** de la barra del navegador. Debe verse así: `https://github.com/antoniomt846-max/mi-dashboard-ventas/blob/main/NOMBRE.ipynb`
5. **Compruébalo:** ábrelo en una ventana de incógnito. Si lo ves sin iniciar sesión, está público.

**Si no se ven las figuras:** vuelve a Colab, ejecuta todo y guarda de nuevo. Si GitHub dice *Unable to render code block*, recarga la página unos segundos después.

---
**Para entregar:** (1) el **enlace público** de tu notebook en GitHub, con tu dashboard final visible, y (2) tus prompts (`prompt_v1`, `prompt_v2` y, si la hiciste, `prompt_v3`) más **tres líneas** sobre qué tuviste que corregir de lo que hizo Gemini y cómo lo pediste (escríbelas en una celda de texto al final del notebook).

## Reflexión sobre el trabajo con Gemini
*(Edita estas tres líneas según lo que realmente pasó al ejecutar tus prompts.)*

1. **Qué corregí:** en la v1 Gemini cambió algunos títulos o cifras respecto a mis tablas; en la v2 le pedí explícitamente *"Conserva exactamente los mismos 6 gráficos y las mismas cifras"* y verifiqué los valores contra `top_productos`, `ing_canal` y `ing_mes_ciudad`.
2. **Qué no me gustó de la v1 y cómo lo pedí:** el pie era difícil de leer al proyectar y los números eran pequeños; en AJUSTES pedí una dona con el total en el centro y tamaños de letra concretos (11 y 10).
3. **Errores de código:** cuando el código devuelto falló, copié el mensaje de error en el siguiente prompt y le pedí que lo corrigiera sin cambiar el contenido de los gráficos.

**Enlace público de mi dashboard:** https://github.com/antoniomt846-max/mi-dashboard-ventas/blob/main/Taller_Graficos.ipynb